# 02 · Risk EDA: tails, cost-index rises, regimes, co-movement

What this notebook answers, for the 3 / 6 / 9-month forecasts and the quote buffer:

0. **Two cost indices**: the PPI-proxy index and the rebuilt market-price index.
1. **Stationarity and tails**: what to model, and why the backtest's 80% bands come out too narrow.
2. **Cost-index rises**: how much the material cost of one unit rose in 3 / 6 / 9 months, and which material drove it.
3. **Regimes**: six periods since 2016 and how volatile each one was.
4. **Co-movement**: how material prices move together, whether that changes in a crisis, and why summing P90s is wrong.

All computations are in `src/eda_risk.py` (the pipeline saves the same tables and figures to `outputs/eda/`); this
notebook shows them and draws the conclusions. Window: Nov-2016 to Sep-2026. A "change" is a log change of a monthly
average price.

> Outputs show challenge data. The pre-commit hook (nbstripout) removes them from every commit.

In [ ]:
import logging
import sys
sys.path.insert(0, "../src")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

import numpy as np
import pandas as pd

import data as D
import eda_risk as R

%matplotlib inline
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)

df, prices, costs = R.load()  # studied series; full preprocessed panel; BOM cost table
pct = lambda t: (100 * t).round(1)

## 0. Two versions of the material cost index

Both use the same fixed weights: kg per unit (standalone BOM workbook `RM_Prices_XHEC_Challenge.xlsx`, which
replaces the older BOM in `data.zip`) × EUR/t in Sep-2026. They differ only in the price series that
moves the steel lines:

- **PPI proxy** (`RM6_NE_IQI_433520`, `RMAirSeT_IIQ_24kV`): carbon and stainless steel follow the EU steel PPI (S09),
  galvanised steel the German steel PPI (S10), the fallback in `bom_materials.csv`.
- **Market prices** (`…_mkt`): the same lines follow the CRU EUR/t prices that set their weight (CR coil, stainless
  304, HDG coil), the BOM file's first choice (L01–L03). Stainless ends Aug-2026, so its last value is carried to
  Sep-2026 (`data.TRAIL_FILL`).

Copper, aluminium and plastics are the same in both.

In [ ]:
t = costs.assign(market_series=[D.MARKET_PROXY.get(m, p) for m, p in zip(costs.material, costs.proxy)])
t[["product", "material", "kg", "eur_per_unit", "cost_share", "proxy", "market_series"]].round(3)

The PPI proxy drives 89% of the AirSeT index (carbon and stainless steel) and 33% of the RM6 index. The
data-quality notebook showed that S09 lags CR coil by about a month and swings only 0.63× as much, so this choice
matters most for AirSeT.

## 1. Stationarity, tails and volatility clustering

### 1.1 Unit roots
ADF tests H0 = unit root, KPSS tests H0 = stationary. "Unit root" = ADF does not reject and KPSS rejects.

In [ ]:
R.stationarity(df).round(3)

- **Price levels have a unit root**: 17 of 22 series clearly; the other 5 (zinc, the three CRU steel prices, EUR/USD)
  are unclear because neither test rejects with ~120 points. No level is clearly stationary.
- **Monthly changes are stationary** for 18 of 22. The 4 borderline ones (copper and aluminium PPIs, both RM6 indices)
  have ADF p = 0.06–0.09 and KPSS does not reject.
- So models work on **log changes**, not levels, as the backtest already does. The P10–P90 range is a range of
  changes from today's price.

### 1.2 Fat tails and volatility clustering

In [ ]:
R.tails(df).round(3)

In [ ]:
R.fig_qq(df);

- **Fat tails, but only in the far tails.** Jarque-Bera rejects normality for 17 of 22 series. On average 3.5% of
  months lie beyond 2.33 sd (normal: 2%); the plastics PPI (excess kurtosis 8.9), Brent (7.0), nickel (5.1) and the
  AirSeT market index (3.6) are the heaviest. But only 15.9% of months lie beyond 1.28 sd (normal: 20%): the
  distributions are peaked in the middle. **Fat tails matter for P95 and the worst case (the buffer), not for an
  80% band.**
- **Volatility clustering** (ARCH-LM p < 0.05) in the steel chain (CRU CR / HDG / stainless, both steel PPIs, iron
  ore), the aluminium PPI, gas, power and both market-price cost indices. **Not** in the exchange-traded metals
  (copper, aluminium, nickel, zinc), Brent or EUR/USD: for those, a calm month says little about the next one.
- Monthly changes of the steel, aluminium and plastics PPIs and of CRU CR / HDG coil are strongly autocorrelated
  (0.6–0.8, last column of §1.3): these series trend, which is why ETS beats naive on the PPIs in the backtest.

### 1.3 Why the backtest's 80% bands are too narrow

Same set-up as the naive model of the backtest: band centred on today's price, width from past 3-month changes,
origins Jan-2020 to Jun-2026, only data known at each origin.

In [ ]:
b = R.band_coverage(df)
display(b.round(2))
b.drop(columns="n").mean().round(3)

- Coverage is **75% on average** (the backtest's 68% also includes the other models). The misses are **lopsided**:
  18% of outcomes end above the band and only 7% below (a correct band: 10% / 10%). 2020-26 was a rising-price
  period, and a band centred on today's price ignores that drift.
- The band is also **too narrow**: the sd of 3-month changes in 2020-26 is about **2× the sd known at the start of
  2020** (CRU CR / HDG coil and the plastics PPI more than 3×). With the realised sd (hindsight) coverage would be 81%.
- Using only the **last 24 months** does not help (73%): volatility rises only after a shock has started, and the
  window forgets the shock in the calm 2023-25 years, just before the 2025-26 moves.

**Conclusion for the forecasts**: the range must be (a) **wider** than recent history suggests, i.e. estimated on
a window that includes stress periods, or floored at the 2020-26 level; (b) **asymmetric**, using empirical
quantiles of past changes, not a symmetric normal band; (c) with **heavier tails** for P95 and beyond.

## 2. How much did the cost of one unit rise?

Every start month from Nov-2016, the change of the cost index over the next h months.

In [ ]:
r = R.cost_rises(df)
for c in ["median", "P90", "P95", "max", "share of windows > +10%"]:
    r[c] = pct(r[c])
r

In [ ]:
R.fig_rises(df);

In [ ]:
R.rise_contributions(prices, costs)  # percentage points of the rise coming from each material, top 3 windows

**Market-price indices** (what a buyer actually pays):

| | 3 months | 6 months | 9 months |
|---|---|---|---|
| RM6: P90 / P95 / max | +12% / +17% / +26% | +23% / +32% / +52% | +32% / +50% / +68% |
| AirSeT: P90 / P95 / max | +14% / +24% / +33% | +22% / +47% / +70% | +33% / +67% / +99% |

- The **PPI-proxy** indices understate this: 3-month P95 +12% (RM6) and +15% (AirSeT), maximum +17–19%.
- The largest rises all start in **late 2020 to early 2022** (reopening rally and war). In a typical window the cost
  hardly moves (median about 0%): a buffer is insurance against those episodes, not a trend.
- **Drivers**: AirSeT rises come almost entirely from steel. Nov-2020 → Feb-2021 (+33%): CR coil +18 pp,
  stainless +14 pp. For RM6, copper is 62% of today's cost, but its largest rises came as much from **stainless
  and carbon steel** as from copper (Nov-2020 → Feb-2021: copper +9 pp, stainless +10 pp, carbon steel +6 pp).
- **For the quote rule**: a 3-month price validity with a buffer at the historical P90 would be about **+12%
  (RM6) / +14% (AirSeT)**, or +17% / +24% at P95. Both need the market-price index: the PPI proxy would suggest
  10% / 9% at P90.

## 3. Regimes

Boundaries set from turning points in the data: COVID low Apr-2020, reopening rally 2021, nickel squeeze Mar-2022,
energy price peak Aug-2022, normalisation 2023-25, copper rally from Oct-2025 (+48% to Sep-2026), oil shock Mar-2026.

In [ ]:
R.fig_regimes(df);

In [ ]:
vol, ratio, n = R.regime_vol(df)
display(n.rename("months"))
R.fig_regime_vol(ratio);

- **2022 was the extreme**: market-price cost indices 2.6× (RM6) and 3.6× (AirSeT) as volatile as in 2017-19; CR
  coil 4.1×, the steel PPIs 3.2–3.4×, gas 3.2×.
- **Volatility has not fully come back down**: in 2023-25 CR coil is still at 1.9× and the AirSeT index at 1.5× the
  2017-19 level. A risk model estimated on the calm years alone would be too narrow, which is what §1.3 found.
- **2025-26 is a trend, not turbulence**, for the BOM: copper rose 48%, yet the cost indices move at 0.9–1.0× their calm
  volatility, because the rise is steady. Volatility is high where the oil shock hits, the plastics PPI (5.5×)
  and Brent (1.8×), and in aluminium (1.7×). For RM6 the risk now is the **level and drift** of copper; for AirSeT
  (with 9% plastics) the oil shock passing into resins.
- 2020 COVID has only 10 months, and the energy and steel series moved differently in it (energy collapsed, CRU
  steel was volatile), so its ratios are noisy.

## 4. How material prices move together

In [ ]:
full, calm, stress = R.correlations(df)
display(full.round(2))
full3, calm3, stress3 = R.correlations(df, h=3)
pd.DataFrame({"monthly changes": [R.mean_pairwise(m) for m in (full, calm, stress)],
              "3-month changes": [R.mean_pairwise(m) for m in (full3, calm3, stress3)]},
             index=["all months", "calm (2017-19, 2023-25)", "stress (2020-22)"]).round(2)

In [ ]:
R.fig_correlation(full, calm, stress, R.rolling_mean_corr(df));

In [ ]:
R.sum_of_quantiles(df, prices, costs).round(3)

- **Two blocks**: non-ferrous metals (copper–aluminium 0.58, nickel–both 0.48) and steel (CR–HDG 0.98, which is
  practically one price; CR–stainless 0.46). Across the blocks, correlations are low (0.14–0.26), and the plastics
  PPI sits in between (0.19–0.37).
- **Correlation rises in a crisis**: the average correlation of the six BOM materials is 0.22 in calm regimes and
  **0.45 in 2020-22** (3-month changes: 0.27 vs **0.62**). Rolling over 24 months it went from about 0.1 in 2020 to
  0.5 in 2021 and again in 2022, fell to below 0.1 in mid-2025 and is back near 0.3 in 2026.
- **Summing P90s is wrong, in both directions.** Over the whole sample, the cost-weighted sum of each material's P90
  overstates the P90 of the cost index by 15% (AirSeT) to 17% (RM6), and the P10 by 23–36%: materials do not all
  hit their extremes in the same quarter. But because correlations double in stress, this diversification is
  smallest exactly when it matters.

**Conclusion for the cost forecast**: the cost P10 / P90 must come from **joint price scenarios**, and those must
keep the stress-time correlation. Resampling **whole historical months (or blocks of months) for all materials
together** keeps the co-movement, the fat tails and the clustering at once. A single fixed correlation matrix
estimated on all months would understate the joint upside in a 2021-22-type episode.

## 5. What this changes downstream

| Finding | Consequence |
|---|---|
| Levels have a unit root, changes are stationary | Forecast log changes; P10–P90 as changes from today's price |
| 80% bands miss mostly upwards; 2020-26 sd ≈ 2× earlier | Wider, asymmetric ranges from empirical quantiles of a window that includes 2020-22 |
| Fat tails beyond P95 | Buffer at P95 / worst case needs the empirical tail, not a normal one |
| PPI proxy halves the steel risk | Business layer (buffer, margin at risk) on the market-price index |
| 3-month buffer: P90 +12% / +14%, P95 +17% / +24% (RM6 / AirSeT) | Starting point for the quote rule |
| Correlation doubles in stress; summing quantiles is off by 15–36% | Joint scenarios by resampling whole months; never sum quantiles |